消息裁剪和删除都会导致上下文缺失，影响回答质量和用户体验。和它们相比，摘要是更适合长会话的折中方案：保语义，不保原文。官方推荐内置SummarizationMiddleware。上

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model_out = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")   
)

model_in = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")   
)


In [2]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# 创建带摘要中间件的 Agent
agent = create_agent(
    model=model_out,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model_in,
            trigger=[
                ("tokens", 100),  # 超过 100 tokens 就摘要
            ],
            keep=("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}",
        )
    ]
)

config = {"configurable": {"thread_id": "1"}}

print("\n进行多轮对话...")
conversations = [
    "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20,  # 强制撑爆 100 tokens
    "请总结一下我的信息"
]

for msg in conversations:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    for msg in response["messages"]:
        msg.pretty_print()
    print("*" * 50)



进行多轮对话...
================================ Human Message =================================

我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...
================================== Ai Message ==================================

你好，张三。你这段话里反复提到了“我是工程师”和“非常长非常长的废话”。

如果你是想让我：
1. **提取关键信息**：你叫张三，是工程师；
2. **压缩总结**：可以概括成“张三，工程师”；
3. **处理长文本**：我也可以帮你做摘要、去重、提炼重点或改写。

你直接告诉我想怎么处理这段内容就行。
**************************************************
================================ Human Message ================================

In [5]:
final_state = agent.get_state(config)
from rich import print as rich_print
rich_print(final_state)


StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='Here is a summary of the conversation to 
date:\n\n用户自我介绍，名叫张三，是一名工程师。信息重复且冗长，主要内容围绕其身份做了多次强调。',
                additional_kwargs={'lc_source': 'summarization'},
                response_metadata={},
                id='6f639ed4-65c5-4b36-8b9a-6124db123783'
            ),
            AIMessage(
                content='你好，张三。你这段话里反复提到了“我是工程师”和“非常长非常长的废话”。\n\n如果你是想让我：\n
1. **提取关键信息**：你叫张三，是工程师；\n2. **压缩总结**：可以概括成“张三，工程师”；\n3. 
**处理长文本**：我也可以帮你做摘要、去重、提炼重点或改写。\n\n你直接告诉我想怎么处理这段内容就行。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 116,
                        'prompt_tokens': 386,
                        'total_tokens': 502,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': 0,
                            'reasoning_tokens': 0,
                            'rejected_prediction_tokens': None,
                            'image_tokens': 0
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': 0,
                            'cached_tokens': 0,
                            'cache_write_tokens': 0,
                            'video_tokens': 0
                        },
                        'cost': 0.0008115,
                        'is_byok': False,
                        'cost_details': {
                            'upstream_inference_cost': 0.0008115,
                            'upstream_inference_prompt_cost': 0.0002895,
                            'upstream_inference_completions_cost': 0.000522
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'openai/gpt-5.4-mini',
                    'system_fingerprint': None,
                    'id': 'gen-1787712241-TSTDyVZ2jNKutBztQuyl',
                    'service_tier': 'default',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a03bf3-ce66-7850-8cdf-5564a4d324f4-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 386,
                    'output_tokens': 116,
                    'total_tokens': 502,
                    'input_token_details': {'audio': 0, 'cache_read': 0},
                    'output_token_details': {'audio': 0, 'reasoning': 0}
                }
            ),
            HumanMessage(
                content='请总结一下我的信息',
                additional_kwargs={},
                response_metadata={},
                id='61f24577-7339-4af3-a5a4-fddef97e7cb7'
            ),
            AIMessage(
                content='你叫**张三**，是一名**工程师**。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 18,
                        'prompt_tokens': 177,
                        'total_tokens': 195,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': 0,
                            'reasoning_tokens': 0,
                            'rejected_prediction_tokens': None,
                            'image_tokens': 0
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': 0,
                            'cached_tokens': 0,
                            'cache_write_tokens': 0,
                            'video_tokens': 0
                        },
                        'cost': 0.00021375,
                        'is_byok': False,
                        '

# 工作原理
```
对话历史: [消息1，消息2，...，消息20] （超过 100 tokens）
↓
SummarizationMiddleware 自动触发
↓
摘要旧消息: "用户是张三，在北京工作，喜欢编程..."
↓
新历史: [摘要，最近消息] （减少到 100 tokens）
```

# 常见问题
## 1.摘要会丢失信息吗？
会有一些细节丢失，但:
- 重要信息会保留（姓名、关键事实）
- 最近的消息完整保留
- 对于大部分场景足够

## 2. 设置最大token数触发摘要的标准是啥？
```
# 模型上下文窗口 4k → 设置 3000
# 模型上下文窗口 8k → 设置 6000
# 模型上下文窗口 16k → 设置 12000

# 留一些余量给工具调用和系统提示
```

## 3.摘要成本高吗？
- 摘要只在超过阈值时触发
- 可以使用便宜的模型（如 gpt‑4o‑mini）
- 相比传输全部历史，通常更便宜

## 4.摘要触发的频率要关注吗？
要关注，根据监控摘要要触发频率，调整阈值。
- 如果频繁触发 → 提高阈值
- 如果从不触发 → 降低阈值